# handling synthetic data


In [1]:
import json
import re
import pandas as pd
from pathlib import Path

folder = Path("/Users/pratyushkumarsethy/Desktop/sem-7/HADL/COURSE-PROJECT/data_and_scripts")

df = pd.read_parquet(
    folder / "or_data.parquet",
    columns=["paragraph", "question", "answer", "wiki_id", "title", "url"],
)

df = df.dropna(subset=["paragraph", "question", "answer", "wiki_id"]).copy()
for column in ["paragraph", "question", "answer"]:
    df = df[df[column].str.strip().ne("")]

# Collect passages reserved for validation and testing.
held_out = []
for split in ["validation", "test"]:
    path = folder / "odia_cleaned" / f"{split}.jsonl"
    held_out += [
        json.loads(line)["passage"]
        for line in path.read_text(encoding="utf-8").splitlines()
    ]

def compact(text):
    return re.sub(r"[\W_]+", "", text)

# An 80-character shared stretch signals substantial passage overlap.
width = 80
stretches = {
    passage[i:i + width]
    for passage in {compact(p) for p in held_out}
    for i in range(len(passage) - width + 1)
}

overlaps = {}
for paragraph in df["paragraph"].unique():
    text = compact(paragraph)
    overlaps[paragraph] = any(
        text[i:i + width] in stretches
        for i in range(0, len(text) - width + 1, 10)
    )

# If one passage overlaps, exclude its whole Wikipedia article.
overlapping_articles = set(
    df.loc[df["paragraph"].map(overlaps), "wiki_id"]
)
df = df[~df["wiki_id"].isin(overlapping_articles)].copy()

def normalise(text):
    return re.sub(r"\s+", " ", text).strip()

# Remove questions that have conflicting answers.
df["_pair"] = (
    df["paragraph"].map(normalise)
    + "\0"
    + df["question"].map(normalise)
)
df["_answer"] = df["answer"].map(normalise)

conflicting = df.groupby("_pair")["_answer"].transform("nunique") > 1
df = df[~conflicting].drop_duplicates("_pair")

df = df.drop(columns=["_pair", "_answer"])
output = folder / "or_synthetic_candidates.parquet"
df.to_parquet(output, index=False)

print("Overlapping Wikipedia articles removed:", len(overlapping_articles))
print("Synthetic candidates remaining:", len(df))
print("Saved to:", output)

Overlapping Wikipedia articles removed: 36
Synthetic candidates remaining: 260643
Saved to: /Users/pratyushkumarsethy/Desktop/sem-7/HADL/COURSE-PROJECT/data_and_scripts/or_synthetic_candidates.parquet


In [2]:
import re
import pandas as pd
from pathlib import Path

folder = Path("/Users/pratyushkumarsethy/Desktop/sem-7/HADL/COURSE-PROJECT/data_and_scripts")
df = pd.read_parquet(folder / "or_synthetic_candidates.parquet")

no_answer_phrases = [
    "ଅଜଣା", "ଉତ୍ତର ଦିଆ", "ତଥ୍ୟ ନାହିଁ",
    "ଜଣା ନାହିଁ", "ପ୍ରଦତ୍ତ ତଥ୍ୟ",
    "ଉଲ୍ଲେଖ ନାହିଁ", "ପାରିବ ନାହିଁ",
]
pattern = "|".join(re.escape(phrase) for phrase in no_answer_phrases)

df = df[
    (df["paragraph"].str.len() >= 30)
    & ~df["answer"].str.contains(pattern, regex=True, na=False)
].copy()

output = folder / "or_synthetic_filtered.parquet"
df.to_parquet(output, index=False)
print("Rows remaining:", len(df))

Rows remaining: 216394
